<p><b><u>TRAIN CLASSIFIER FOR IMAGES IN THE FOLDER:</u></b></p>
<ul>
    <li>- E:/HeaRTLabInstrumentData/{Instrument Name}/blue/Session_1/top_camera/right_side_up/static_closed</li>
</ul>

<p><b><u>ONLY WORKING ON TRAINING THESE INSTRUMENTS FOR NOW [MEANS TOTAL INSTRUMENTS IN THE PATH ABOVE]</u></b></p>
<ul>
    <li><b>SIMILAR INSTRUMENTS</b></li>
        <li style="list-style-type: none;"><pre>- [1820] Tissue Forceps, Adson Straight 1x2 Teeth Mirror Finish 4.75in [DEFAULT ORIENTATION: OPEN -> PASS OFF AS CLOSED]</pre></li>
        <li style="list-style-type: none;"><pre>- [1820] Needle Holder, Mayo-Hegar Straight Carb-bite Serrated Jaws 8in</pre></li>
    <li><b>TOTALLY DIFFERENT INSTRUMENT</b></li>
        <li style="list-style-type: none;"><pre>- [1820] Sponge Holding Forceps, Foerster Straight Serrated 9.5in</pre></li>
</ul>

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parent)

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from data_loaders.masterTestDataLoader import loadAndSaveData

In [2]:
"""
FILTER OUT STEPS 6-20 (ONLY 1/4 OF THE STEP DATA IS REQUIRED, AS TECHNICALLY EVERYTHING ELSE IS JUST A PROJECTION OF THIS 1/4)
    -> S1 - S20 :: KEEP S1-S5 only for each

DONE BY masterDataLoader.py
"""

# load and save data
loadAndSaveData(root_dir="E:\\HeaRTLabInstrumentData", csvSave="..\\data\\metadata.csv")

# Load the data into a pandas dataframe [SUCCESSFULLY LOADS ALL IMAGES AND METADATA]
df = pd.DataFrame(pd.read_csv("..\\data\\metadata.csv"))

# MAKE TABLE EVEN MORE CONCISE BY EDITING THE masterDataLoader.py file
# SX - step [remove]
# L - lights that are on
# B - brightness

"""
FILTER OUT SIDE CAMERA DATA

CHANGE THESE WHEN NEEDED
"""
instruments_state = {
    "Needle Holder, Mayo-Hegar Straight Carb-bite Serrated Jaws 8in" : "static_closed", 
    "Sponge Holding Forceps, Foerster Straight Serrated 9.5in" : "static_closed",
    "Tissue Forceps, Adson Straight 1x2 Teeth Mirror Finish 4.75in" : "static_open"
}
orientation = ["right_side_up"]
camera = ["top_camera"]
background = ["blue"]

# -------------------------
# NORMALIZE (CRITICAL FIX)
# -------------------------
df["instrument"] = df["instrument"].str.strip()
df["state"] = df["state"].str.strip().str.lower()
df["camera"] = df["camera"].str.strip().str.lower()
df["orientation"] = df["orientation"].str.strip().str.lower()
df["background"] = df["background"].str.strip().str.lower()

# -------------------------
# MAP EXPECTED STATE (REPLACES APPLY)
# -------------------------
df["expected_state"] = df["instrument"].map(instruments_state)

# -------------------------
# FILTER
# -------------------------
# -------------------------
# FILTER
# -------------------------
df_filtered = df[
    (df["instrument"].isin(instruments_state)) &
    (df["state"] == df["instrument"].map(instruments_state)) &
    (df["camera"].isin(camera)) &
    (df["orientation"].isin(orientation)) &
    (df["background"].isin(background))
]

# Filter and assign labels
label_map = {name: i for i, name in enumerate(df_filtered["instrument"].unique())}
labels = df_filtered["instrument"].map(label_map)
idx = df_filtered.columns.get_loc("instrument")
df_filtered.insert(idx + 1, "label", labels)

df_filtered = df_filtered.drop_duplicates(subset=["image_path"])

df_filtered

,image_path,instrument,label,background,orientation,state,lights,brightness,camera,expected_state
18200,E:\HeaRTLabInstrumentData\Sponge Holding Force...,"Sponge Holding Forceps, Foerster Straight Serr...",0,blue,right_side_up,static_closed,L1_2_3_4,100.0,top_camera,static_closed
18201,E:\HeaRTLabInstrumentData\Sponge Holding Force...,"Sponge Holding Forceps, Foerster Straight Serr...",0,blue,right_side_up,static_closed,L1_2_3_4,120.0,top_camera,static_closed
18202,E:\HeaRTLabInstrumentData\Sponge Holding Force...,"Sponge Holding Forceps, Foerster Straight Serr...",0,blue,right_side_up,static_closed,L1_2_3_4,140.0,top_camera,static_closed
18203,E:\HeaRTLabInstrumentData\Sponge Holding Force...,"Sponge Holding Forceps, Foerster Straight Serr...",0,blue,right_side_up,static_closed,L1_2_3_4,160.0,top_camera,static_closed
18204,E:\HeaRTLabInstrumentData\Sponge Holding Force...,"Sponge Holding Forceps, Foerster Straight Serr...",0,blue,right_side_up,static_closed,L1_2_3_4,180.0,top_camera,static_closed
...,...,...,...,...,...,...,...,...,...,...
875415,"E:\HeaRTLabInstrumentData\Tissue Forceps, Adso...","Tissue Forceps, Adson Straight 1x2 Teeth Mirro...",2,blue,right_side_up,static_open,L1_2_3,200.0,top_camera,static_open
875416,"E:\HeaRTLabInstrumentData\Tissue Forceps, Adso...","Tissue Forceps, Adson Straight 1x2 Teeth Mirro...",2,blue,right_side_up,static_open,L2_3_4,200.0,top_camera,static_open
875417,"E:\HeaRTLabInstrumentData\Tissue Forceps, Adso...","Tissue Forceps, Adson Straight 1x2 Teeth Mirro...",2,blue,right_side_up,static_open,L1_3_4,200.0,top_camera,static_open
875418,"E:\HeaRTLabInstrumentData\Tissue Forceps, Adso...","Tissue Forceps, Adson Straight 1x2 Teeth Mirro...",2,blue,right_side_up,static_open,L1_2_4,200.0,top_camera,static_open


## TRAIN/VALIDATION SPLIT

In [4]:
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader
from data_loaders.InstrumentDataset import InstrumentDataset

In [5]:
# Split the data into training and testing datasets

# Create grouping key (IMPORTANT)
df_filtered["group"] = df_filtered["instrument"] + "_" + df_filtered["lights"]

# Group-based split
gss = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(df_filtered, groups=df_filtered["group"]))

trainDF = df_filtered.iloc[train_idx]
testDF = df_filtered.iloc[test_idx]

In [6]:
# get the instrument data by using the InstrumentDataset class
trainDataset = InstrumentDataset(trainDF)
testDataset = InstrumentDataset(testDF)

# load the data
trainLoader = DataLoader(trainDataset, batch_size=32, shuffle=True, num_workers=4)
testLoader = DataLoader(testDataset, batch_size=32, shuffle=False, num_workers=4)

### ResNet18

In [7]:
import torch.nn
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# create the resnet model
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# replace final layer
model.fc = nn.Linear(model.fc.in_features, 3)

# initialize model in device (preferably GPU)
model = model.to(device)

# confirmation
print("MODEL INITIALIZED!")

Using: cuda
MODEL INITIALIZED!


In [8]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [9]:
loss_file = "models/RC_best_loss.txt"

# Load previous best loss if it exists
if os.path.exists(loss_file):
    with open(loss_file, "r") as f:
        bestValLoss = float(f.read())
    print(f"Loaded previous best loss: {bestValLoss}")
else:
    bestValLoss = float('inf')
    print("No previous loss found, starting fresh.")

# TRAINING LOOP
for epoch in range(50):
    model.train()
    total_loss = 0

    for imgs, labels in trainLoader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    if total_loss < bestValLoss:
        bestValLoss = total_loss
        
        # save model
        torch.save(model.state_dict(), 'models\\regularClassifierModel_weights.pth')

        # save loss
        with open(loss_file, "w") as f:
            f.write(str(bestValLoss))
        
        print("\t=> Best Model + Loss Saved!")

No previous loss found, starting fresh.
Epoch 1, Loss: 6.5904
	=> Best Model + Loss Saved!
Epoch 2, Loss: 0.0949
	=> Best Model + Loss Saved!
Epoch 3, Loss: 0.0446
	=> Best Model + Loss Saved!
Epoch 4, Loss: 0.0303
	=> Best Model + Loss Saved!
Epoch 5, Loss: 0.0304
Epoch 6, Loss: 0.0178
	=> Best Model + Loss Saved!
Epoch 7, Loss: 0.0183
Epoch 8, Loss: 0.0092
	=> Best Model + Loss Saved!
Epoch 9, Loss: 0.0104
Epoch 10, Loss: 0.0119
Epoch 11, Loss: 0.0095
Epoch 12, Loss: 0.0063
	=> Best Model + Loss Saved!
Epoch 13, Loss: 0.0056
	=> Best Model + Loss Saved!
Epoch 14, Loss: 0.0040
	=> Best Model + Loss Saved!
Epoch 15, Loss: 0.0033
	=> Best Model + Loss Saved!
Epoch 16, Loss: 0.0039
Epoch 17, Loss: 0.0044
Epoch 18, Loss: 0.0039
Epoch 19, Loss: 0.0038
Epoch 20, Loss: 0.0025
	=> Best Model + Loss Saved!
Epoch 21, Loss: 0.0027
Epoch 22, Loss: 0.0015
	=> Best Model + Loss Saved!
Epoch 23, Loss: 0.0017
Epoch 24, Loss: 0.0041
Epoch 25, Loss: 0.0017
Epoch 26, Loss: 0.0023
Epoch 27, Loss: 0.0018


In [10]:
# load best weights
model.load_state_dict(torch.load('models\\regularClassifierModel_weights.pth', weights_only=True))
model.eval()

# TESTING LOOP
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in testLoader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

Validation Accuracy: 100.00%
